# Paper Reproduction Demo

This notebook demonstrates the SMLR workflow for surrogate model construction,
matching the methodology from the paper. We use synthetic test data to keep
the notebook self-contained.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from smlr import Surrogate, StrengthDataset, StrengthSample
from smlr.metrics import normalized_l2

## Generate Synthetic Training Data

We create a synthetic dataset mimicking the structure of nuclear strength functions.
Each spectrum is a sum of Lorentzians whose parameters depend on input parameters.

In [ ]:
def lorentzian(e, e0, gamma, s):
    """Single Lorentzian peak."""
    return s * gamma / np.pi / ((e - e0)**2 + gamma**2)

def generate_spectrum(params, energy, rng):
    """Generate a synthetic spectrum as sum of Lorentzians."""
    alpha, beta = params
    # Pole positions and strengths depend on parameters
    e1 = 5.0 + 2.0 * alpha
    e2 = 12.0 + 3.0 * beta
    s1 = 0.5 + 0.3 * alpha
    s2 = 0.8 - 0.2 * beta
    gamma = 1.5
    
    spectrum = lorentzian(energy, e1, gamma, s1) + lorentzian(energy, e2, gamma, s2)
    # Add small noise
    spectrum += rng.normal(0, 0.01, len(energy))
    return np.maximum(spectrum, 0)  # Ensure non-negative

# Create training samples on a parameter grid
rng = np.random.default_rng(42)
energy = np.linspace(0, 25, 200)

alpha_vals = np.linspace(0, 1, 5)
beta_vals = np.linspace(0, 1, 5)

samples = []
for alpha in alpha_vals:
    for beta in beta_vals:
        params = np.array([alpha, beta])
        strength = generate_spectrum(params, energy, rng)
        samples.append(StrengthSample(params, energy, strength))

dataset = StrengthDataset(samples)
print(f"Created dataset with {len(dataset.samples)} samples")
print(f"Parameter shape: {dataset.parameters().shape}")

## Visualize Training Data

Let's look at how the spectra vary with parameters.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Show spectra varying alpha (beta=0.5)
ax = axes[0]
for sample in dataset.samples:
    if np.isclose(sample.params[1], 0.5):
        ax.plot(sample.energy, sample.strength, label=f"α={sample.params[0]:.2f}")
ax.set_xlabel("Energy")
ax.set_ylabel("Strength")
ax.set_title("Varying α (β=0.5)")
ax.legend()

# Show spectra varying beta (alpha=0.5)
ax = axes[1]
for sample in dataset.samples:
    if np.isclose(sample.params[0], 0.5):
        ax.plot(sample.energy, sample.strength, label=f"β={sample.params[1]:.2f}")
ax.set_xlabel("Energy")
ax.set_ylabel("Strength")
ax.set_title("Varying β (α=0.5)")
ax.legend()

plt.tight_layout()
fig

## Train Surrogate Model

We use the unified `Surrogate` class with the regression backend.
This fits Lorentzian mixtures to each training spectrum, then learns
how the mixture parameters depend on the input parameters.

In [ ]:
# Train with different numbers of poles
for n_poles in [5, 7, 10]:
    model = Surrogate(
        "pmm",
        n_poles=n_poles,
        retain=0.8
    )
    model.fit(dataset)
    print(f"n_poles={n_poles}: fitted {len(dataset.samples)} samples")

## Leave-One-Out Cross-Validation

To evaluate the emulator quality, we perform leave-one-out cross-validation:
train on all samples except one, predict at the held-out point, measure error.

In [ ]:
def leave_one_out_error(dataset, n_poles, test_idx):
    """Compute LOO error for a single held-out sample."""
    # Split
    train_samples = [s for i, s in enumerate(dataset.samples) if i != test_idx]
    test_sample = dataset.samples[test_idx]
    train_ds = StrengthDataset(train_samples)
    
    # Train with PMM
    model = Surrogate("pmm", n_poles=n_poles, retain=0.8)
    model.fit(train_ds)
    
    # Predict
    result = model.predict(test_sample.params, test_sample.energy)
    
    # Error
    return normalized_l2(result.spectrum, test_sample.strength, test_sample.energy)

# Evaluate for different n_poles
n_poles_list = [5, 7, 10, 12]
test_indices = [0, 6, 12, 18, 24]  # Sample of test points

results = {}
for np_ in n_poles_list:
    errors = [leave_one_out_error(dataset, np_, idx) for idx in test_indices]
    results[np_] = np.mean(errors)
    print(f"n_poles={np_}: mean LOO error = {results[np_]:.4f}")

best_np = min(results, key=results.get)
print(f"\nBest n_poles: {best_np} (error={results[best_np]:.4f})")

## Prediction at New Parameter Point

Finally, we train on all data and predict at a new (interpolated) parameter point.

In [ ]:
# Train final model
model = Surrogate("pmm", n_poles=best_np, retain=0.8)
model.fit(dataset)

# Predict at new point
new_params = np.array([0.35, 0.65])  # Interpolation point
result = model.predict(new_params, energy)

# Compare to ground truth
true_spectrum = generate_spectrum(new_params, energy, rng)
error = normalized_l2(result.spectrum, true_spectrum, energy)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(energy, true_spectrum, 'k-', lw=2, label='Ground truth')
ax.plot(energy, result.spectrum, 'r--', lw=2, label='Surrogate prediction')
ax.set_xlabel('Energy')
ax.set_ylabel('Strength')
ax.set_title(f'Prediction at (α={new_params[0]}, β={new_params[1]}), error={error:.4f}')
ax.legend()

print(f"\nNormalized L2 error: {error:.4f}")
fig

## Conclusion

This notebook demonstrated:
1. Creating a `StrengthDataset` from synthetic data
2. Training a `Surrogate` model with the PMM (Parametric Matrix Model) backend
3. Leave-one-out cross-validation to select `n_poles`
4. Prediction at new parameter points

For real applications, replace the synthetic data generation with your
actual strength function calculations.